# BFF Camera Demo - Live demo notebook (Google Colab)

Boots the FastAPI backend + NiceGUI frontend inside this Colab runtime
and exposes them via an ngrok tunnel. The user opens the printed URL
in their browser to interact with the demo.

This is the v0.1 cloud path; the production deliverable is the
Windows desktop installer (`dist/BFFDemo-Setup-0.1.0.exe`).

In [ ]:
# 0. Environment setup - clone the repo, install deps.
# Colab Secrets: REPO_URL (e.g. https://github.com/<org>/<repo>.git) and,
# for a private repo, GITHUB_TOKEN (scope: repo).
from google.colab import userdata

REPO_URL = userdata.get('REPO_URL')
try:
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    TOKEN = ''
CLONE_URL = (REPO_URL if not TOKEN
             else REPO_URL.replace('https://', f'https://x-access-token:{TOKEN}@'))
!git clone {CLONE_URL} /content/camera-ai 2>/dev/null || (cd /content/camera-ai && git pull)
# Order matters: the `ml` package needs bff-backend, which is not on PyPI.
%pip install -e /content/camera-ai/backend[cuda,cpu]
%pip install -e /content/camera-ai/ml
%pip install -e /content/camera-ai/frontend[dev]
%pip install pyngrok


In [ ]:
import os
from getpass import getpass

os.environ['NGROK_AUTHTOKEN'] = getpass('ngrok authtoken: ')
os.environ['SUPABASE_URL'] = os.environ.get('SUPABASE_URL', 'https://naltduhojaazpodznvry.supabase.co')
os.environ['SUPABASE_SERVICE_ROLE_KEY'] = getpass('Supabase service role: ')
os.environ['S3_ENDPOINT_URL'] = os.environ.get('S3_ENDPOINT_URL', 'https://naltduhojaazpodznvry.supabase.co/storage/v1/s3')
os.environ['S3_REGION'] = 'ap-southeast-1'
os.environ['S3_ACCESS_KEY_ID'] = getpass('S3 access: ')
os.environ['S3_SECRET_ACCESS_KEY'] = getpass('S3 secret: ')
os.environ['MODE'] = 'colab'
os.environ['INFERENCE_PROVIDER'] = 'cuda'
os.environ['BACKEND_PORT'] = '8001'
os.environ['FRONTEND_PORT'] = '3001'
os.environ['DATABASE_MODE'] = 'public'   # use 'offline' to run without Supabase


In [ ]:
# Start the BE in the background.
%cd /content/camera-ai
import subprocess, time, os
be_log = open('be.log', 'w')
be = subprocess.Popen(
    ['python', '-m', 'backend', '--host', '0.0.0.0', '--port', '8001'],
    env=os.environ.copy(), stdout=be_log, stderr=subprocess.STDOUT)
print('BE pid:', be.pid)
time.sleep(5)
import httpx
print('BE healthz:', httpx.get('http://localhost:8001/healthz').json())

In [ ]:
# Open the ngrok tunnels BEFORE starting the frontend:
# the frontend needs the public backend URL.
from pyngrok import ngrok
ngrok.set_auth_token(os.environ['NGROK_AUTHTOKEN'])
be_tunnel = ngrok.connect(8001, 'http')
fe_tunnel = ngrok.connect(3001, 'http')
print(f'\n  Backend URL:  {be_tunnel.public_url}')
print(f'  Frontend URL: {fe_tunnel.public_url}')


In [ ]:
# Start the FE in the background, pointing at the public backend.
import subprocess, time, os
fe_log = open('fe.log', 'w')
fe = subprocess.Popen(
    ['python', '-m', 'frontend', '--host', '0.0.0.0', '--port', '3001',
     '--backend-url', be_tunnel.public_url, '--no-browser'],
    env=os.environ.copy(), stdout=fe_log, stderr=subprocess.STDOUT)
print('FE pid:', fe.pid)
time.sleep(5)
import httpx
print('FE /:', httpx.get('http://localhost:3001/').status_code)
print('\n  Open the Frontend URL in your browser. The first request takes ~10s '
      'as the tunnel hands out the certificate.')

try:
    input('\n  Press Enter to stop the demo...')
except KeyboardInterrupt:
    pass
finally:
    ngrok.kill()
    fe.terminate(); be.terminate()
    fe.wait(timeout=5); be.wait(timeout=5)
    print('stopped.')
